# Autoencoder, Variational Autoencoder, and a Mean-less VAE Ablation — MNIST

## Aim
To implement and compare three latent-variable image models on MNIST digits:

1. A vanilla **Autoencoder (AE)** — a deterministic encoder–decoder trained only to reconstruct its input.
2. A standard **Variational Autoencoder (VAE)** whose encoder predicts *both* a mean **μ** and a variance **σ²** for the approximate posterior, combined with the reparameterization trick and the KL-divergence regularizer.
3. A **mean-less VAE** ablation — the "final" model — whose encoder predicts *only* a variance and never learns a mean, so every latent code is drawn from `N(0, σ²)` instead of `N(μ, σ²)`.

## Objectives

1. Build one shared convolutional/linear encoder–decoder backbone so the three models differ only in how the latent code is produced.
2. Implement the plain Autoencoder and train it with a reconstruction-only objective.
3. Implement the full VAE: encoder outputs `(μ, logσ²)`, sample `z = μ + σ ⊙ ε` with `ε ~ N(0, I)`, and optimize the evidence lower bound (reconstruction loss + KL divergence).
4. Implement the mean-less VAE by fixing `μ = 0` and learning only `logσ²`, so `z = σ ⊙ ε`, and show that the general KL formula degenerates to the variance-only term automatically when `μ ≡ 0`.
5. Compare reconstruction quality, generated samples, and 2‑D latent-space structure across all three models, and discuss why removing the learned mean hurts the model.

## Dataset

```text
Dataset: MNIST handwritten digits
Dataset source: torchvision.datasets.MNIST (downloaded automatically on first run)
Image size: 28 x 28, grayscale, pixel values scaled to [0, 1]
Training samples: 60,000
Test samples: 10,000
```

No manual download is required — `torchvision` fetches and caches the files under `CONFIG["data_root"]` the first time the notebook runs.

## Setup

Install dependencies if needed: `pip install torch torchvision matplotlib numpy`.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

CONFIG = {
    "data_root": "./data",
    "batch_size": 128,
    "latent_dim": 2,      # kept at 2 so the latent space can be plotted directly
    "hidden_dim": 256,
    "epochs": 15,
    "learning_rate": 1e-3,
}

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root=CONFIG["data_root"], train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root=CONFIG["data_root"], train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

print(f"Training samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")


def show_grid(images, title, n=8):
    fig, axes = plt.subplots(1, n, figsize=(n * 1.2, 1.5))
    for i in range(n):
        axes[i].imshow(images[i].squeeze(), cmap="gray")
        axes[i].axis("off")
    fig.suptitle(title)
    plt.show()


sample_images, sample_labels = next(iter(train_loader))
show_grid(sample_images, "Sample MNIST digits")

## Part 1 — Vanilla Autoencoder

The encoder maps an image straight to a single latent vector `z`; the decoder maps `z` back to an image. There is no
distribution, no sampling, and no regularizer — only a reconstruction loss, so the model is free to memorize whatever
mapping minimizes it.

In [ ]:
class Encoder(nn.Module):
    """Deterministic encoder: image -> single latent vector."""

    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.out = nn.Linear(hidden_dim // 2, latent_dim)

    def forward(self, x):
        h = self.net(x)
        return self.out(h)


class Decoder(nn.Module):
    """Shared decoder for all three models: latent vector -> image."""

    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 28 * 28),
            nn.Sigmoid(),
        )

    def forward(self, z):
        x = self.net(z)
        return x.view(-1, 1, 28, 28)


class Autoencoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.encoder = Encoder(latent_dim, hidden_dim)
        self.decoder = Decoder(latent_dim, hidden_dim)

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

In [ ]:
def train_autoencoder(model, loader, epochs, lr):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for images, _ in loader:
            images = images.to(DEVICE)
            optimizer.zero_grad()
            recon, _ = model(images)
            loss = F.binary_cross_entropy(recon, images, reduction="sum") / images.size(0)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(loader.dataset)
        history.append(epoch_loss)
        print(f"[AE] Epoch {epoch:02d}/{epochs} | reconstruction loss: {epoch_loss:.4f}")
    return history


ae_model = Autoencoder(CONFIG["latent_dim"], CONFIG["hidden_dim"])
ae_history = train_autoencoder(ae_model, train_loader, CONFIG["epochs"], CONFIG["learning_rate"])

In [ ]:
plt.plot(ae_history)
plt.xlabel("Epoch")
plt.ylabel("Reconstruction loss (summed BCE per sample)")
plt.title("Autoencoder training loss")
plt.show()

In [ ]:
def visualize_reconstructions(model, loader, title, n=8):
    model.eval()
    images, _ = next(iter(loader))
    images = images.to(DEVICE)
    with torch.no_grad():
        outputs = model(images)
        recon = outputs[0]
    images, recon = images.cpu(), recon.cpu()

    fig, axes = plt.subplots(2, n, figsize=(n * 1.2, 3))
    for i in range(n):
        axes[0, i].imshow(images[i].squeeze(), cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(recon[i].squeeze(), cmap="gray")
        axes[1, i].axis("off")
    axes[0, 0].set_title("Original", loc="left")
    axes[1, 0].set_title("Reconstructed", loc="left")
    fig.suptitle(title)
    plt.show()


visualize_reconstructions(ae_model, test_loader, "Autoencoder reconstructions")

## Part 2 — Variational Autoencoder (mean *and* variance)

The encoder now outputs the parameters of a Gaussian, `μ` and `log σ²`, instead of a single point. A latent code is
drawn with the **reparameterization trick**:

```
z = μ + σ ⊙ ε,   ε ~ N(0, I),   σ = exp(0.5 · log σ²)
```

which keeps sampling differentiable with respect to `μ` and `logσ²`. The training objective is the negative ELBO:

```
L = reconstruction_loss(x, x̂)  +  KL( N(μ, σ²) ‖ N(0, I) )
KL = -0.5 · Σ ( 1 + log σ² − μ² − σ² )
```

`μ` lets the encoder place each input at a different location in latent space; `σ²` controls how much noise is mixed
into that location. Both terms are needed for the KL divergence above.

In [ ]:
class VAEEncoder(nn.Module):
    """Encoder that predicts a mean and a log-variance for the approximate posterior."""

    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)

    def forward(self, x):
        h = self.net(x)
        return self.fc_mu(h), self.fc_logvar(h)


class VAE(nn.Module):
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim, hidden_dim)
        self.decoder = Decoder(latent_dim, hidden_dim)

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, z, mu, logvar


def vae_loss(recon, x, mu, logvar):
    """Reconstruction + KL( N(mu, exp(logvar)) || N(0, I) ), averaged per sample.

    When mu is identically zero (the mean-less ablation in Part 3) the mu**2
    term vanishes on its own and this reduces to the variance-only KL term.
    """
    recon_loss = F.binary_cross_entropy(recon, x, reduction="sum") / x.size(0)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon_loss, kl_loss

In [ ]:
def train_vae(model, loader, epochs, lr, tag="VAE"):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"recon": [], "kl": [], "total": []}
    for epoch in range(1, epochs + 1):
        model.train()
        running = {"recon": 0.0, "kl": 0.0, "total": 0.0}
        for images, _ in loader:
            images = images.to(DEVICE)
            optimizer.zero_grad()
            recon, z, mu, logvar = model(images)
            recon_loss, kl_loss = vae_loss(recon, images, mu, logvar)
            loss = recon_loss + kl_loss
            loss.backward()
            optimizer.step()

            bs = images.size(0)
            running["recon"] += recon_loss.item() * bs
            running["kl"] += kl_loss.item() * bs
            running["total"] += loss.item() * bs

        n = len(loader.dataset)
        for key in history:
            history[key].append(running[key] / n)
        print(
            f"[{tag}] Epoch {epoch:02d}/{epochs} | recon: {history['recon'][-1]:.4f} "
            f"| KL: {history['kl'][-1]:.4f} | total: {history['total'][-1]:.4f}"
        )
    return history


vae_model = VAE(CONFIG["latent_dim"], CONFIG["hidden_dim"])
vae_history = train_vae(vae_model, train_loader, CONFIG["epochs"], CONFIG["learning_rate"], tag="VAE")

In [ ]:
plt.plot(vae_history["recon"], label="Reconstruction")
plt.plot(vae_history["kl"], label="KL divergence")
plt.plot(vae_history["total"], label="Total (negative ELBO)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("VAE training losses")
plt.legend()
plt.show()

visualize_reconstructions(vae_model, test_loader, "VAE reconstructions")

In [ ]:
def plot_latent_space(model, loader, title):
    model.eval()
    zs, labels = [], []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(DEVICE)
            mu, _ = model.encoder(images)
            zs.append(mu.cpu())
            labels.append(targets)
    zs = torch.cat(zs).numpy()
    labels = torch.cat(labels).numpy()

    plt.figure(figsize=(6, 5))
    scatter = plt.scatter(zs[:, 0], zs[:, 1], c=labels, cmap="tab10", s=4, alpha=0.6)
    plt.colorbar(scatter, ticks=range(10), label="digit")
    plt.xlabel("z[0]")
    plt.ylabel("z[1]")
    plt.title(title)
    plt.show()


def generate_samples(model, n, title):
    model.eval()
    with torch.no_grad():
        z = torch.randn(n, CONFIG["latent_dim"], device=DEVICE)
        samples = model.decoder(z).cpu()
    fig, axes = plt.subplots(1, n, figsize=(n * 1.2, 1.5))
    for i in range(n):
        axes[i].imshow(samples[i].squeeze(), cmap="gray")
        axes[i].axis("off")
    fig.suptitle(title)
    plt.show()


plot_latent_space(vae_model, test_loader, "VAE latent space (encoder mean, colored by digit)")
generate_samples(vae_model, n=8, title="VAE samples decoded from z ~ N(0, I)")

## Part 3 — Final model: VAE with no mean consideration (variance only)

This ablation removes the mean branch entirely: the encoder predicts only `log σ²`, and `μ` is fixed at zero for
every input, so

```
z = 0 + σ ⊙ ε = σ ⊙ ε
```

Substituting `μ ≡ 0` into the general KL formula from Part 2 makes the `μ²` term disappear on its own:

```
KL = -0.5 · Σ ( 1 + log σ² − 0 − σ² ) = -0.5 · Σ ( 1 + log σ² − σ² )
```

so `vae_loss` from Part 2 can be reused unchanged — it already degenerates correctly.

**Why this should hurt the model:** `μ` is what lets the encoder *place* different inputs at different points in
latent space; `σ²` only controls how much noise is mixed into that point. With `μ ≡ 0`, every digit is encoded
around the same origin and can only be distinguished by how spread out its noise is, not by direction or position.
Expect the reconstructions to blur across digit classes, the generated samples to look like an average digit rather
than a specific one, and the latent scatter plot to lose the class clusters visible in Part 2.

In [ ]:
class MeanlessVAEEncoder(nn.Module):
    """Predicts only log-variance; the latent mean is fixed at zero (no learned mean branch)."""

    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)

    def forward(self, x):
        h = self.net(x)
        logvar = self.fc_logvar(h)
        mu = torch.zeros_like(logvar)  # mean is never learned, always zero
        return mu, logvar


class MeanlessVAE(nn.Module):
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.encoder = MeanlessVAEEncoder(latent_dim, hidden_dim)
        self.decoder = Decoder(latent_dim, hidden_dim)

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std  # mu is always zero, so z = std * eps

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, z, mu, logvar


meanless_model = MeanlessVAE(CONFIG["latent_dim"], CONFIG["hidden_dim"])
meanless_history = train_vae(
    meanless_model, train_loader, CONFIG["epochs"], CONFIG["learning_rate"], tag="Mean-less VAE"
)

In [ ]:
plt.plot(meanless_history["recon"], label="Reconstruction")
plt.plot(meanless_history["kl"], label="KL divergence")
plt.plot(meanless_history["total"], label="Total (negative ELBO)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Mean-less VAE training losses")
plt.legend()
plt.show()

visualize_reconstructions(meanless_model, test_loader, "Mean-less VAE reconstructions")
plot_latent_space(meanless_model, test_loader, "Mean-less VAE latent space (encoder mean is always 0)")
generate_samples(meanless_model, n=8, title="Mean-less VAE samples decoded from z ~ N(0, I)")

## Comparison

Side-by-side final losses and reconstructions for all three models.

In [ ]:
print(f"{'Model':35s} final loss")
print(f"{'-' * 35} ----------")
print(f"{'Autoencoder (reconstruction only)':35s} {ae_history[-1]:.4f}")
print(f"{'VAE (mean + variance)':35s} {vae_history['total'][-1]:.4f}  "
      f"(recon {vae_history['recon'][-1]:.4f} + KL {vae_history['kl'][-1]:.4f})")
print(f"{'Mean-less VAE (variance only)':35s} {meanless_history['total'][-1]:.4f}  "
      f"(recon {meanless_history['recon'][-1]:.4f} + KL {meanless_history['kl'][-1]:.4f})")

In [ ]:
def compare_reconstructions(models, names, loader, n=8):
    images, _ = next(iter(loader))
    images = images.to(DEVICE)

    rows = len(models) + 1
    fig, axes = plt.subplots(rows, n, figsize=(n * 1.2, rows * 1.5))
    for i in range(n):
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap="gray")
        axes[0, i].axis("off")
    axes[0, 0].set_title("Original", loc="left")

    for row, (model, name) in enumerate(zip(models, names), start=1):
        model.eval()
        with torch.no_grad():
            recon = model(images)[0]
        for i in range(n):
            axes[row, i].imshow(recon[i].cpu().squeeze(), cmap="gray")
            axes[row, i].axis("off")
        axes[row, 0].set_title(name, loc="left")

    plt.tight_layout()
    plt.show()


compare_reconstructions(
    [ae_model, vae_model, meanless_model],
    ["Autoencoder", "VAE (mean + var)", "Mean-less VAE (var only)"],
    test_loader,
)

## Conclusion

- The **Autoencoder** reaches the lowest reconstruction loss because it optimizes reconstruction alone, with no
  constraint on the latent distribution. Its latent space has no guaranteed structure, so it cannot be sampled from
  to generate new digits.
- The **VAE** trades a little reconstruction accuracy for a latent space regularized toward `N(0, I)`. Because the
  encoder learns a distinct `μ` per input, digits of the same class cluster together and different classes occupy
  different regions, which is what makes sampling `z ~ N(0, I)` and decoding it produce recognizable digits.
- The **mean-less VAE** removes exactly the one mechanism (`μ`) that lets the encoder place inputs at different
  latent locations. With `μ ≡ 0`, class identity can only be expressed through the predicted variance, which is a
  single non-negative scalar per dimension rather than a position — a much weaker signal. This should show up as
  higher reconstruction loss, reconstructions that look like a generic/blurred digit rather than the input digit,
  and a latent scatter plot with the class clusters collapsed around the origin, confirming that the *mean* — not
  the variance — is what carries most of the useful information about the input.